# 🎬 GAPT-CINEMA: Faststart & GPU Optimizer (Google Colab)
สมุดบันทึกสำหรับประมวลผลไฟล์หนัง **"ธี่หยด สมิงเขาขวาง"** จาก 3.79 GB (2 Parts) ให้อยู่ที่ **~1.7 GB (Single File)** ด้วย **NVIDIA GPU T4** พร้อมโครงสร้าง **Faststart** เพื่อให้เล่นบน Smart TV และเว็บได้ทันทีแบบไม่ต้องรอโหลด

> ⚡ **วิธีใช้งานง่าย ๆ แค่ 1 คลิก:** กดที่เมนูด้านบน **Runtime (รันไทม์)** → **Run all (เรียกใช้ทั้งหมด)** หรือกดแป้นพิมพ์ **Ctrl + F9**

### 1️⃣ ตรวจสอบ GPU และติดตั้งโมดูลที่จำเป็น

In [ ]:
!nvidia-smi
!pip install -q telethon cryptg requests

### 2️⃣ ดาวน์โหลดไฟล์ต้นฉบับและแปลงด้วย NVIDIA GPU (เสร็จใน ~2-3 นาที)

In [ ]:
import os, time, subprocess

STREAM_URL = "https://stream.capt-th.work/stream/4381915212/2"
INPUT_FILE = "input_raw.mp4"
OUTPUT_FILE = "tee_yod_2_faststart.mp4"

print("📥 [1/2] กำลังดาวน์โหลดไฟล์จาก Server (ความเร็วเน็ต Colab 1-2 Gbps)...")
t0 = time.time()
!curl -L --retry 3 -o {INPUT_FILE} "{STREAM_URL}"
t1 = time.time()

input_mb = os.path.getsize(INPUT_FILE) / (1024 * 1024)
print(f"✅ ดาวน์โหลดเสร็จใน {t1 - t0:.1f} วินาที! ขนาด: {input_mb:.1f} MB ({input_mb/1024:.2f} GB)")

# ตรวจสอบ GPU Hardware Acceleration
has_gpu = False
try:
    res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if res.returncode == 0:
        has_gpu = True
except Exception:
    pass

print(f"⚡ [2/2] เริ่มการประมวลผลวิดีโอ (GPU Hardware Mode: {has_gpu})...")
if has_gpu:
    # ใช้ NVIDIA NVENC Hardware Encoder: ความเร็วสูงมาก (~2-3 นาทีสำหรับหนัง 1.5 ชม.)
    # บิตเรต 2000k เพื่อให้ขนาดเหลือ ~1.7 GB (ต่ำกว่าโควต้า 2.0 GB ของ Telegram) + Faststart moov atom
    !ffmpeg -y -i {INPUT_FILE} \
      -c:v h264_nvenc -preset p4 -b:v 2000k -maxrate 2400k -bufsize 4000k \
      -c:a aac -b:a 128k \
      -movflags +faststart \
      {OUTPUT_FILE}
else:
    # Fallback CPU ถ้าผู้ใช้ไม่ได้เปิด GPU
    !ffmpeg -y -i {INPUT_FILE} \
      -c:v libx264 -preset veryfast -b:v 2000k -maxrate 2400k -bufsize 4000k \
      -c:a aac -b:a 128k \
      -movflags +faststart \
      {OUTPUT_FILE}

t2 = time.time()
out_size = os.path.getsize(OUTPUT_FILE)
out_mb = out_size / (1024 * 1024)
print(f"\n🎉 แปลงไฟล์เสร็จสิ้นใน {t2 - t1:.1f} วินาที!")
print(f"📦 ขนาดไฟล์ผลลัพธ์: {out_mb:.1f} MB ({out_size / (1024**3):.2f} GB)")
if out_size < 1950 * 1024 * 1024:
    print("✅ ยอดเยี่ยม! ขนาดไฟล์ต่ำกว่า 1.95 GB พร้อมส่งเข้า Telegram เป็นไฟล์เดี่ยว 100%!")
else:
    print("⚠️ ขนาดเกิน 1.95 GB กรุณาลดบิตเรต")

### 3️⃣ ส่งไฟล์เข้า Telegram Channel และอัปเดตระบบ GAPT-CINEMA อัตโนมัติ

In [ ]:
import asyncio, json, time, requests
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo, DocumentAttributeFilename

API_ID = 33156484
API_HASH = "d23bc5cc206aeb76929cb3ff6481a5eb"
BOT_TOKEN = "8627312320:AAGw9_C7okMMGMYj0VBRYfv80ppXBVtZdAo"
CHANNEL_ID = -1004381915212
OUTPUT_FILE = "tee_yod_2_faststart.mp4"
MOVIE_TITLE = "ธี่หยด สมิงเขาขวาง.mp4"

async def upload():
    print("🚀 กำลังเชื่อมต่อ Telegram Bot...")
    async with TelegramClient('colab_session', API_ID, API_HASH) as client:
        await client.start(bot_token=BOT_TOKEN)
        out_size = os.path.getsize(OUTPUT_FILE)
        print(f"📤 กำลังส่งไฟล์ '{MOVIE_TITLE}' ({out_size / (1024*1024):.1f} MB) เข้า Telegram...")
        
        def progress(current, total):
            pct = (current / total) * 100
            print(f"\rกำลังส่ง: {pct:.1f}% ({current/1024/1024:.1f}/{total/1024/1024:.1f} MB)", end="")

        caption_meta = {
            "v": 2,
            "name": MOVIE_TITLE,
            "size": out_size,
            "mime": "video/mp4",
            "uploadedAt": int(time.time() * 1000),
            "totalParts": 1,
            "faststart": True
        }

        msg = await client.send_file(
            CHANNEL_ID,
            OUTPUT_FILE,
            caption=json.dumps(caption_meta),
            supports_streaming=True,
            attributes=[
                DocumentAttributeVideo(duration=6672, w=1920, h=1080, supports_streaming=True),
                DocumentAttributeFilename(file_name=MOVIE_TITLE)
            ],
            progress_callback=progress
        )
        print(f"\n\n🎉 อัปโหลดสำเร็จแล้ว 100%!")
        print(f"📌 Message ID ใหม่บน Telegram: {msg.id}")
        
        # แจ้งระบบ GAPT-CINEMA ให้อัปเดตฐานข้อมูลและเปิดดูได้ทันที
        stream_url = f"https://stream.capt-th.work/watch/4381915212/{msg.id}"
        try:
            requests.get(stream_url, timeout=5)
            print(f"✅ บันทึกลง GAPT-CINEMA เรียบร้อย!")
        except Exception:
            pass
        
        print(f"📺 ดูหนังออนไลน์ทันที: {stream_url}")
        print("\n💡 ไฟล์ต้นฉบับ 2 ส่วนเดิมบน OCI SSD สามารถลบทิ้งได้ตามต้องการ!")

await upload()